### Imports

In [1]:
import pandas as pd

### Lading Data

In [3]:
df_raw = pd.read_csv('../data/flights_sample_3m.csv')


### Inspecting Data

In [4]:
print(df_raw.columns)

df_raw.head()



Index(['FL_DATE', 'AIRLINE', 'AIRLINE_DOT', 'AIRLINE_CODE', 'DOT_CODE',
       'FL_NUMBER', 'ORIGIN', 'ORIGIN_CITY', 'DEST', 'DEST_CITY',
       'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF',
       'WHEELS_ON', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY',
       'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME',
       'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'DELAY_DUE_CARRIER',
       'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY',
       'DELAY_DUE_LATE_AIRCRAFT'],
      dtype='object')


,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,...,DIVERTED,CRS_ELAPSED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT
0,2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,1562,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",...,0.0,186.0,176.0,153.0,1065.0,NaN,NaN,NaN,NaN,NaN
1,2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1149,MSP,"Minneapolis, MN",SEA,"Seattle, WA",...,0.0,235.0,236.0,189.0,1399.0,NaN,NaN,NaN,NaN,NaN
2,2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,459,DEN,"Denver, CO",MSP,"Minneapolis, MN",...,0.0,118.0,112.0,87.0,680.0,NaN,NaN,NaN,NaN,NaN
3,2023-03-06,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,2295,MSP,"Minneapolis, MN",SFO,"San Francisco, CA",...,0.0,260.0,285.0,249.0,1589.0,0.0,0.0,24.0,0.0,0.0
4,2020-02-23,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,407,MCO,"Orlando, FL",DFW,"Dallas/Fort Worth, TX",...,0.0,181.0,182.0,153.0,985.0,NaN,NaN,NaN,NaN,NaN


### Key Columns

In [ ]:

# Must not-null columns
must_cols  =['FL_DATE','AIRLINE_CODE','ORIGIN','DEST','CANCELLED','DIVERTED']
must_cols_null_rate = df_raw[must_cols].isna().mean().sort_values(ascending=False)
print(f'must_cols_null_rate:\n{must_cols_null_rate}')

must_cols_null_rate:
FL_DATE         0.0
AIRLINE_CODE    0.0
ORIGIN          0.0
DEST            0.0
CANCELLED       0.0
DIVERTED        0.0
dtype: float64


In [9]:
# Conditional not-null columns
cond_cols = ['ARR_DELAY']
mask_active = (df_raw['CANCELLED'] == 0) & (df_raw['DIVERTED'] == 0)
df_raw.loc[mask_active, cond_cols].isna().mean().sort_values(ascending=False)

ARR_DELAY    6.863880e-07
dtype: float64

In [11]:
key_cols = ['FL_DATE','AIRLINE_CODE','FL_NUMBER','ORIGIN','DEST','CRS_DEP_TIME']
flight_key = (
    df_raw['FL_DATE'].astype(str) + '_' +
    df_raw['AIRLINE_CODE'].astype(str) + '_' +
    df_raw['FL_NUMBER'].astype(str) + '_' +
    df_raw['ORIGIN'].astype(str) + '_' +
    df_raw['DEST'].astype(str) + '_' +
    df_raw['CRS_DEP_TIME'].astype(str)
)

dup_rate = flight_key.duplicated().mean()
print(f'flight_key_dup_rate: {dup_rate:.4%}')

print(f'flight_key_dup_counts:\n{flight_key.value_counts().head(10)}')

flight_key_dup_rate: 0.0000%
flight_key_dup_counts:
2021-05-08_AA_2565_ORD_MIA_1345    1
2023-01-03_AA_1456_SLC_DFW_1619    1
2020-12-28_YX_5804_SDF_DTW_1300    1
2023-04-23_DL_1208_ORD_DTW_1030    1
2020-10-16_WN_1289_ATL_LAS_1710    1
2019-11-11_NK_569_DTW_ATL_1930     1
2019-11-17_MQ_3650_BZN_DFW_1338    1
2023-03-13_WN_3197_OAK_LAS_800     1
2022-10-13_AS_731_SEA_SFO_1335     1
2022-03-18_OO_3776_BIS_MSP_525     1
Name: count, dtype: int64


### Data Quality

In [17]:
print('>>> ARR_DELAY describe:')
print(df_raw['ARR_DELAY'].describe())

sanity = df_raw['ARR_DELAY'].dropna()

print('>>> ARR_DELAY percentiles:')
print(sanity.quantile([0.9, 0.95, 0.99, 0.999]))


print('>>> ARR_DELAY IQR Bounds:')
q1 = df_raw['ARR_DELAY'].quantile(0.25)
q3 = df_raw['ARR_DELAY'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
print(f'ARR_DELAY IQR bounds: [{lower_bound:.2f}, {upper_bound:.2f}]')

print('>>> ARR_DELAY Min and Max:')
print(f'Min: {sanity.min():.2f}, Max: {sanity.max():.2f}')


>>> ARR_DELAY describe:
count    2.913802e+06
mean     4.260858e+00
std      5.117482e+01
min     -9.600000e+01
25%     -1.600000e+01
50%     -7.000000e+00
75%      7.000000e+00
max      2.934000e+03
Name: ARR_DELAY, dtype: float64
>>> ARR_DELAY percentiles:
0.900     36.0
0.950     71.0
0.990    189.0
0.999    643.0
Name: ARR_DELAY, dtype: float64
>>> ARR_DELAY IQR Bounds:
ARR_DELAY IQR bounds: [-50.50, 41.50]
>>> ARR_DELAY Min and Max:
Min: -96.00, Max: 2934.00


In [18]:
# Sanity Check
rate_severe = (sanity > 189).mean()
rate_extreme = (sanity > 643).mean()
rate_early = (sanity < -60).mean()

print(f'ARR_DELAY severe delay rate (>189 min): {rate_severe:.4%}')
print(f'ARR_DELAY extreme delay rate (>643 min): {rate_extreme:.4%}')
print(f'ARR_DELAY early arrival rate (<-60 min): {rate_early:.4%}')

ARR_DELAY severe delay rate (>189 min): 0.9993%
ARR_DELAY extreme delay rate (>643 min): 0.0999%
ARR_DELAY early arrival rate (<-60 min): 0.0192%


In [22]:
df_raw['CANCELLED'].value_counts(normalize=True).head()

CANCELLED
0.0    0.97362
1.0    0.02638
Name: proportion, dtype: float64

In [23]:
df_raw['DIVERTED'].value_counts(normalize=True).head()

DIVERTED
0.0    0.997648
1.0    0.002352
Name: proportion, dtype: float64

In [24]:
df_raw['ORIGIN'].str.len().value_counts().head()

ORIGIN
3    3000000
Name: count, dtype: int64

In [25]:
df_raw['DEST'].str.len().value_counts().head()

DEST
3    3000000
Name: count, dtype: int64